In [76]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Deterministic Sankey from method

In [91]:
# Generate sbm and adjacency matrix
import sys
sys.path.append('..')
from pygraphon.graphons import Graphon
from pygraphon.graphons import StepGraphon
from experiments.utils_generators import AssociativeFullyRandom
k = 2
n = 400
# theta init
theta_intra_min, theta_intra_max = 0.4, 0.8
theta_inter_min, theta_inter_max = 0.1, 0.3
import numpy as np
np.random.seed(420)
theta = AssociativeFullyRandom(
                    K=k,
                    min_inside_classes=theta_intra_min,
                    max_inside_classes=theta_intra_max,
                    min_outside_classes=theta_inter_min,
                    max_outside_classes=theta_inter_max,
                )
graphon = StepGraphon(theta.Theta, bandwidthHist=1 / k)
adjacency_matrix=graphon.draw(n=n, exchangeable=False)
print(graphon.get_theta())

[[0.74726659 0.17200129]
 [0.17200129 0.65189141]]


In [92]:
#import smoothnethist
from pygraphon.estimators import SmoothNetHist

est = SmoothNetHist(bandwidth=1/(2*k))
est.fit(adjacency_matrix)

In [93]:
import numpy as np
list_graphons = [est.all_cluster_graphons[i].get_theta()[np.triu_indices(est.all_cluster_graphons[i].get_theta().shape[0])] for i in range(len(est.all_cluster_graphons))]

sort_indices = np.argsort(list_graphons[-1])

for i in range(len(list_graphons)):
    list_graphons[i] = list_graphons[i][sort_indices]
graphons_array = np.array(list_graphons)
graphons_array = np.rot90(np.insert(graphons_array, 0, np.ones(graphons_array.shape[1]), axis=0), k=3)

In [94]:
#makes list into pandas dataframe
import pandas as pd
pd.set_option('display.float_format', '{:.3f}'.format)
columns = []
for i in range(len(list_graphons)):
    columns.append("SmoothNetHist_" + str(len(list_graphons)-i))
columns.append("Count")

def list_to_df(array, columns):
    df = pd.DataFrame(array, columns=columns)
    return df

df = list_to_df(graphons_array, columns=columns)
df

,SmoothNetHist_10,SmoothNetHist_9,SmoothNetHist_8,SmoothNetHist_7,SmoothNetHist_6,SmoothNetHist_5,SmoothNetHist_4,SmoothNetHist_3,SmoothNetHist_2,SmoothNetHist_1,Count
0,0.129,0.129,0.129,0.129,0.129,0.129,0.174,0.174,0.174,0.435,1.000
1,0.177,0.183,0.183,0.189,0.189,0.189,0.174,0.174,0.174,0.435,1.000
2,0.188,0.183,0.183,0.189,0.189,0.189,0.174,0.174,0.174,0.435,1.000
3,0.201,0.201,0.201,0.189,0.189,0.189,0.174,0.174,0.174,0.435,1.000
4,0.627,0.627,0.634,0.634,0.634,0.634,0.634,0.634,0.704,0.435,1.000
5,0.642,0.642,0.634,0.634,0.634,0.634,0.634,0.634,0.704,0.435,1.000
6,0.700,0.700,0.700,0.700,0.711,0.711,0.711,0.739,0.704,0.435,1.000
7,0.722,0.722,0.722,0.722,0.711,0.711,0.711,0.739,0.704,0.435,1.000
8,0.749,0.749,0.749,0.749,0.749,0.767,0.767,0.739,0.704,0.435,1.000
9,0.785,0.785,0.785,0.785,0.785,0.767,0.767,0.739,0.704,0.435,1.000


# Plotting sankey

In [95]:
import sys
sys.path.insert(0, r"c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages")
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [96]:
def find_peak_index(lst):
    for i in range(1, len(lst)):
        if lst[i] < lst[i - 1]:
            return i - 1
    return len(lst) - 1

In [97]:
source_nodes = [0.129, 0.177, 0.188,0.201,0.627,0.642,0.700,0.722,0.749,0.785, 0.183, 0.634, 0.189, 0.711, 0.767, 0.174, 0.739, 0.704, 0.435]
idx = find_peak_index(source_nodes)
# the 0.0001 is to avoid the error of having nodes one the boundaries of [0,1], somehow it messes up the visualization
source_intervals = [0.0001 if i < idx+1 else (i-idx)/len(source_nodes[idx+1:]) for i in range(len(source_nodes))]
nodes = [{"label": str(label)+'_'+str(i)} for i, label in enumerate(sorted(source_nodes))]
intervals_x = [interval for _, interval in sorted(zip(source_nodes, source_intervals))]
intervals_x[intervals_x.index(1)] -= 0.0001
intervals_y = [i/len(nodes) for i in range(len(nodes))]
intervals_y[intervals_y.index(0)] += 0.0001
print(nodes)

[{'label': '0.129_0'}, {'label': '0.174_1'}, {'label': '0.177_2'}, {'label': '0.183_3'}, {'label': '0.188_4'}, {'label': '0.189_5'}, {'label': '0.201_6'}, {'label': '0.435_7'}, {'label': '0.627_8'}, {'label': '0.634_9'}, {'label': '0.642_10'}, {'label': '0.7_11'}, {'label': '0.704_12'}, {'label': '0.711_13'}, {'label': '0.722_14'}, {'label': '0.739_15'}, {'label': '0.749_16'}, {'label': '0.767_17'}, {'label': '0.785_18'}]


In [98]:
links = [
    {'source': 2, 'target': 3, 'value': 1},
    {'source': 4, 'target': 3, 'value': 1},
    {'source': 8, 'target': 9, 'value': 1},
    {'source': 10, 'target': 9, 'value': 1},
    {'source': 3, 'target': 5, 'value': 2},
    {'source': 6, 'target': 5, 'value': 1},
    {'source': 11, 'target': 13, 'value': 1},
    {'source': 14, 'target': 13, 'value': 1},
    {'source': 16, 'target': 17, 'value': 1},
    {'source': 18, 'target': 17, 'value': 1},
    {'source': 0, 'target': 1, 'value': 1},
    {'source': 5, 'target': 1, 'value': 3},
    {'source': 13, 'target': 15, 'value': 2},
    {'source': 17, 'target': 15, 'value': 2},
    {'source': 9, 'target': 12, 'value': 2},
    {'source': 15, 'target': 12, 'value': 4},
    {'source': 12, 'target': 7, 'value': 6},
    {'source': 1, 'target': 7, 'value': 4},
]

In [99]:
import plotly.graph_objects as go
# Define the data

#sorted_links = sorted(links, key=lambda k: k['source']) #this doesnt chnage anything
# Create the figure and subplot
fig = make_subplots(rows=1, cols=1)

# Add the Sankey trace
fig.add_trace(go.Sankey(
    arrangement = "perpendicular",
    node=dict(
        pad=10*len(nodes),  # Adjust the padding between nodes
        thickness=20,  # Adjust the thickness of nodes
        line=dict(color='black', width=1),  # Define the color and width of node lines
        label=[f"{float(node['label']):.3f}" for node in nodes],  # Extract node labels from the 'nodes' list
        x = intervals_x,
        y = intervals_y,
    ),
    link=dict(
        source=[link['source'] for link in links],  # Extract source indices from the 'links' list
        target=[link['target'] for link in links],  # Extract target indices from the 'links' list
        value=[link['value'] for link in links],  # Extract values from the 'links' list
    )
))

# Update the layout
fig.update_layout(title_text='Sankey Diagram')

# Show the plot
fig.show()

